# Setup — Create Silver Lakehouse Tables
Creates all Silver 1 (staging) and Silver 2 (combined) Delta tables in the Lakehouse.

Safe to re-run — uses `mode('ignore')` so existing tables and data are never touched.

**Run once** after provisioning the Lakehouse, before triggering any pipeline DAGs.

In [ ]:
import json
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, DateType, TimestampType
)

### Parameters

In [ ]:
try:
    env = env  # noqa: F821
except NameError:
    env = "dev"

### Config

In [ ]:
sys.path.insert(0, "/lakehouse/default/Files/shared")
import shared_functions as gf

config_path = Path(__file__).parent.parent.parent.parent / "config" / f"{env}.yaml"
import yaml
cfg: dict = yaml.safe_load(config_path.read_text())

WORKSPACE = cfg["workspace_name"]
LAKEHOUSE = cfg["lakehouse"]

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

print(f"[setup/create_silver_tables] env={env} workspace={WORKSPACE} lakehouse={LAKEHOUSE}")

### Helper

In [ ]:
def create_table(table_name: str, schema: StructType) -> None:
    """Write an empty DataFrame with the given schema as a Delta table.
    mode('ignore') means: create if not exists, skip if it already exists.
    """
    path = gf.onelake_abfss(WORKSPACE, LAKEHOUSE, f"Tables/silver/{table_name}")
    (
        spark.createDataFrame([], schema)
        .write
        .format("delta")
        .mode("ignore")
        .save(path)
    )
    print(f"  OK  {table_name}")

### Silver 1 — Staging tables
Minimal schema with metadata columns only. Data columns are added automatically
by schema evolution when `clean.ipynb` writes the first real batch.

In [ ]:
print("[1/2] Silver 1 staging tables")

# Metadata columns common to all staging tables
meta = [
    StructField("_source_name", StringType(), True),
    StructField("_source_file", StringType(), True),
    StructField("_run_id",      StringType(), True),
]

create_table("staging_fedex_direct_invoice", StructType([
    StructField("invoice_date",    DateType(),   True),
    StructField("ship_date",       DateType(),   True),
    StructField("charge_amount",   DoubleType(), True),
    StructField("fuel_surcharge",  DoubleType(), True),
    StructField("tax_amount",      DoubleType(), True),
] + meta))

create_table("staging_dhl_direct_invoice", StructType([
    StructField("invoice_date",  DateType(),   True),
    StructField("pickup_date",   DateType(),   True),
    StructField("charge_amount", DoubleType(), True),
    StructField("vat_amount",    DoubleType(), True),
] + meta))

create_table("staging_ups_carrier_report", StructType([
    StructField("invoice_date",    DateType(),   True),
    StructField("billed_amount",   DoubleType(), True),
    StructField("adjusted_amount", DoubleType(), True),
] + meta))

create_table("staging_usps_tracking", StructType([
    StructField("scan_date",     DateType(), True),
    StructField("delivery_date", DateType(), True),
] + meta))

create_table("staging_fedex_rate_card", StructType([
    StructField("effective_date",     DateType(),   True),
    StructField("expiry_date",        DateType(),   True),
    StructField("base_rate",          DoubleType(), True),
    StructField("fuel_surcharge_pct", DoubleType(), True),
] + meta))


### Silver 2 — Combined tables
Full canonical schemas matching the output of `combine.ipynb`.

In [ ]:
print("[2/2] Silver 2 combined tables")

# carrier_invoice — unified invoice table across all carriers
create_table("carrier_invoice", StructType([
    StructField("invoice_number",      StringType(),    True),
    StructField("invoice_date",        DateType(),      True),
    StructField("carrier",             StringType(),    True),
    StructField("purchase_contract",   StringType(),    True),
    StructField("tracking_number",     StringType(),    True),
    StructField("shipment_date",       DateType(),      True),
    StructField("origin_country",      StringType(),    True),
    StructField("destination_country", StringType(),    True),
    StructField("weight_kg",           DoubleType(),    True),
    StructField("charge_amount",       DoubleType(),    True),
    StructField("charge_currency",     StringType(),    True),
    StructField("service_type",        StringType(),    True),
    StructField("account_number",      StringType(),    True),
    StructField("_source_name",        StringType(),    True),
    StructField("_source_file",        StringType(),    True),
    StructField("_run_id",             StringType(),    True),
]))

# carrier_tracking — unified tracking events across all carriers
create_table("carrier_tracking", StructType([
    StructField("tracking_number",  StringType(),    True),
    StructField("carrier",          StringType(),    True),
    StructField("scan_date",        DateType(),      True),
    StructField("scan_event",       StringType(),    True),
    StructField("scan_location",    StringType(),    True),
    StructField("delivery_date",    DateType(),      True),
    StructField("delivery_status",  StringType(),    True),
    StructField("_source_name",     StringType(),    True),
    StructField("_source_file",     StringType(),    True),
    StructField("_run_id",          StringType(),    True),
]))

# carrier_rate_card — rate schedules across all carriers
create_table("carrier_rate_card", StructType([
    StructField("carrier",             StringType(),    True),
    StructField("service_type",        StringType(),    True),
    StructField("zone",                StringType(),    True),
    StructField("weight_break",        DoubleType(),    True),
    StructField("base_rate",           DoubleType(),    True),
    StructField("fuel_surcharge_pct",  DoubleType(),    True),
    StructField("effective_date",      DateType(),      True),
    StructField("expiry_date",         DateType(),      True),
    StructField("_source_name",        StringType(),    True),
    StructField("_source_file",        StringType(),    True),
    StructField("_run_id",             StringType(),    True),
]))

In [ ]:
print("\nAll silver tables created (or already existed — no data was changed).")
mssparkutils.notebook.exit("done")  # noqa: F821